# Common

TODO: DMSO_ts should have ts as VALIDATION

In [4]:
import collections
import itertools
import os

import numpy as np
import scanpy as sc

# DMSO
un_treatment_list = [f'DMSO_{t}hr' for t in [48, 48, 48, 48, 48, 3, 6, 12, 24]]
pert_treatment_list = [f'Tram_{t}hr' for t in [3, 6, 12, 24, 48, 3, 6, 12, 24]]

# Untreated
un_treatment_list += 5 * ['Untreated_48hr']
pert_treatment_list += [f'Tram_{t}hr' for t in [3, 6, 12, 24, 48]]

# 6hr
# un_treatment_list = ['DMSO_6hr']
# pert_treatment_list = ['Tram_6hr']

In [2]:
# Normalization function
def normalize_counts(counts, stats=None, sample_count=10_000, log1p=True):
    # Sample count normalization
    if sample_count is not None:
        counts = sample_count * counts / counts.sum(axis=-1, keepdims=True)

    # Log transformation
    if log1p:
        counts = np.log1p(counts)

    # Get standardization parameters
    if stats is None:
        mean = counts.mean(axis=0, keepdims=True)
        std = counts.std(axis=0, keepdims=True)
        std = np.where(std == 0, 1, std)
    else:
        mean, std = stats
    
    # Standardize
    counts = (counts - mean) / std

    return counts, (mean, std)

# Mean MSE difference between perturbed and predicted distributions
def mean_mse_diff(perturbed, predicted):
    perturbed_mean = perturbed.mean(axis=0)
    predicted_mean = predicted.mean(axis=0)
    mse_diff = np.square(perturbed_mean - predicted_mean).mean()

    return mse_diff

# Pearson correlation between mean changes in true and predicted distributions
def pearson_delta(unperturbed_true, perturbed_true, perturbed_pred, unperturbed_pred=None):
    # Defaults
    unperturbed_pred = unperturbed_true if unperturbed_pred is None else unperturbed_pred

    # Compute deltas
    true_delta = perturbed_true.mean(axis=0) - unperturbed_true.mean(axis=0)
    pred_delta = perturbed_pred.mean(axis=0) - unperturbed_pred.mean(axis=0)
    pdelta = np.corrcoef(true_delta, pred_delta)[0, 1]

    return pdelta

# Wasserstein distance between perturbed and predicted distributions
def wasserstein_distance(perturbed, predicted):
    # Scipy method
    # from scipy.stats import wasserstein_distance_nd
    # return wasserstein_distance_nd(perturbed, predicted)

    # OT method
    import ot
    a, b = ot.utils.unif(perturbed.shape[0]), ot.utils.unif(predicted.shape[0])
    M_raw = ot.dist(perturbed, predicted, metric='sqeuclidean')
    OT_mat = ot.emd(a, b, M_raw / M_raw.max(), numItermax=1_000_000)
    
    return np.sqrt((OT_mat * M_raw).sum())

# CellTRIP

In [ ]:
# Load data
import celltrip
adata_ref, = celltrip.utility.processing.read_adatas('s3://nkalafut-celltrip/DrugSeries/expression.h5ad', backed=True)
adata_ref.obs['train'] = adata_ref.obs['train_rand']
adata_ref_obs = adata_ref.obs
def get_data(un_treatment=None, pert_treatment=None):
    # Copy adata
    adata = adata_ref
    adata.obs = adata_ref_obs.copy()

    # Subset by un_treatment and pert_treatment
    if un_treatment is not None and pert_treatment is not None:
        adata = adata[adata.obs['treatment'].isin([un_treatment, pert_treatment])]

    # Also mark random dmso_24 as train
    # train_tram_24 = adata.obs['train_dmso_24hr'] & ~adata.obs['train']
    # train_dmso_24 = (adata.obs['treatment'] == 'DMSO_24hr') * (np.random.rand(adata.shape[0]) < .8)
    # adata.obs['train_w_24'] = adata.obs['train'] | adata.obs['train_dmso_24hr'] 

    # Merge train and train_24
    # adata.obs['train'] = adata.obs['train'] if pert_treatment != 'Tram_24hr' else adata.obs['train_w_24']

    return adata

# Export subset data
for un_treatment, pert_treatment in itertools.product(un_treatment_list, pert_treatment_list):
    adata = get_data(un_treatment, pert_treatment)
    adata.write_h5ad(f'../plots/drugseries/tram_data_{un_treatment}_{pert_treatment}.h5ad')

# Export all data
adata = get_data()
adata.write_h5ad(f'../plots/drugseries/tram_data_all.h5ad')

# GEARS

In [ ]:
import pickle

import gears

In [ ]:
from gears.utils import create_cell_graph_dataset_for_prediction

# Use multi-perturbation with only validation set and no mean, used for Wasserstein distance calculation
class GEARS(gears.GEARS):
    def predict(self, pert_list):
        """
        Predict the transcriptome given a list of genes/gene combinations being
        perturbed

        Parameters
        ----------
        pert_list: list
            list of genes/gene combiantions to be perturbed

        Returns
        -------
        results_pred: dict
            dictionary of predicted transcriptome
        results_logvar: dict
            dictionary of uncertainty score

        """
        ## given a list of single/combo genes, return the transcriptome
        ## if uncertainty mode is on, also return uncertainty score.
        
        # self.ctrl_adata = self.adata[self.adata.obs['condition'] == 'ctrl']
        self.ctrl_adata = self.adata[(self.adata.obs['condition'] == 'ctrl') & (~self.adata.obs['train'].astype(bool))]
        for pert in pert_list:
            for i in pert:
                if i not in self.pert_list:
                    raise ValueError(i+ " is not in the perturbation graph. "
                                        "Please select from GEARS.pert_list!")
        
        if self.config['uncertainty']:
            results_logvar = {}
            
        self.best_model = self.best_model.to(self.device)
        self.best_model.eval()
        results_pred = {}
        results_logvar_sum = {}
        
        from torch_geometric.data import DataLoader
        for pert in pert_list:
            try:
                #If prediction is already saved, then skip inference
                results_pred['_'.join(pert)] = self.saved_pred['_'.join(pert)]
                if self.config['uncertainty']:
                    results_logvar_sum['_'.join(pert)] = self.saved_logvar_sum['_'.join(pert)]
                continue
            except:
                pass
            
            cg = create_cell_graph_dataset_for_prediction(pert, self.ctrl_adata,
                                                    self.pert_list, self.device)
            # loader = DataLoader(cg, 300, shuffle = False)
            loader = DataLoader(cg, self.ctrl_adata.shape[0], shuffle = False)
            batch = next(iter(loader))
            batch.to(self.device)

            with torch.no_grad():
                if self.config['uncertainty']:
                    p, unc = self.best_model(batch)
                    results_logvar['_'.join(pert)] = np.mean(unc.detach().cpu().numpy(), axis = 0)
                    results_logvar_sum['_'.join(pert)] = np.exp(-np.mean(results_logvar['_'.join(pert)]))
                else:
                    p = self.best_model(batch)
                    
            # results_pred['_'.join(pert)] = np.mean(p.detach().cpu().numpy(), axis = 0)
            results_pred['_'.join(pert)] = p.detach().cpu().numpy()
                
        self.saved_pred.update(results_pred)
        
        if self.config['uncertainty']:
            self.saved_logvar_sum.update(results_logvar_sum)
            return results_pred, results_logvar_sum
        else:
            return results_pred

In [ ]:
for (un_treatment, pert_treatment), i in itertools.product(zip(un_treatment_list, pert_treatment_list), range(5)):
    # Seed
    import torch
    torch.manual_seed(42 + i)

    # Read and format for GEARS
    adata = sc.read_h5ad(f'../plots/drugseries/tram_data_{un_treatment}_{pert_treatment}.h5ad')
    # adata = adata[adata.obs['Training']]  # Filter to training
    adata.obs['cell_type'] = adata.obs['cell_line']
    conditions = {
        un_treatment: 'ctrl',
        pert_treatment: 'MAP2K1+MAP2K2'}
    adata = adata[adata.obs['treatment'].isin(list(conditions.keys()))]
    adata.obs['condition'] = adata.obs['treatment'].cat.rename_categories(conditions)

    # Preprocess
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    # sc.pp.scale(adata)  # Unused per author recommendations
    # sc.pp.highly_variable_genes(adata, n_top_genes=5000, subset=True)  # Added based on author recs, but removed for MAP2K1 and MAP2K2
    pert_data = gears.PertData('../plots/drugseries')
    pert_data.new_data_process(dataset_name='tram_gears', adata=adata)
    pert_data.load(data_path = '../plots/drugseries/tram_gears')

    # Split and get dataloader
    adata.obs['split'] = 'val'
    adata.obs.loc[adata.obs['train'], 'split'] = 'train'
    set2conditions = dict(adata.obs.groupby('split').agg({'condition': lambda x: x}).condition)
    set2conditions = {i: j.unique().tolist() for i,j in set2conditions.items()}
    set2conditions['val'] = set2conditions['test'] = set2conditions['val']  # We limit val ourselves
    split_fname = '../plots/drugseries/tram_gears/splits/custom.pkl'
    pickle.dump(set2conditions, open(split_fname, 'wb'))
    # {'train': ['ctrl', 'MAP2K1+MAP2K2'], 'val': ['ctrl', 'MAP2K1+MAP2K2']}
    pert_data.prepare_split(split='custom', split_dict_path=split_fname, seed=42+i)
    pert_data.get_dataloader(batch_size=32, test_batch_size=128)

    # Compute mean and std
    counts = adata[adata.obs['split'] == 'train'].X.toarray()
    mean = counts.mean(axis=0, keepdims=True)
    std = counts.std(axis=0, keepdims=True)
    std = np.where(std == 0, 1, std)

    # Train model
    gears_model = GEARS(pert_data, device='cuda:0')  # gears.GEARS for default
    gears_model.model_initialize(hidden_size=32)
    try:
        gears_model.train(epochs=20)  # KeyError: 'SKIN_ctrl_1' ??
    except KeyError as e:
        print(f'KeyError encountered: {e}')

    # Save model
    # model_fname = f'../plots/drugseries/tram_gears/model_{un_treatment}_{pert_treatment}_{i}'
    # gears_model.save_model(model_fname)
    # gears_model.load_pretrained(model_fname)

    # Predict perturbation
    pert = gears_model.predict([['MAP2K1', 'MAP2K2']])['MAP2K1_MAP2K2']

    # Invert normalization 05/29/2026
    # pert = (pert * adata.var['std'].values) + adata.var['mean'].values
    # pert = np.expm1(pert)

    # Save
    np.save(f'../plots/drugseries/GEARS_perturbation_{un_treatment}_{pert_treatment}_{i}.npy', (pert - mean) / std)
    np.save(f'../plots/drugseries/True_un_{un_treatment}_{pert_treatment}_{i}.npy', (adata[adata.obs['treatment'] == un_treatment].X.toarray() - mean) / std)
    np.save(f'../plots/drugseries/True_pert_{un_treatment}_{pert_treatment}_{i}.npy', (adata[adata.obs['treatment'] == pert_treatment].X.toarray() - mean) / std)


In [8]:
# Load each result sequentially
results = []
for (un_treatment, pert_treatment), i in itertools.product(zip(un_treatment_list, pert_treatment_list), range(5)):
    # Load perturbed result
    try:
        perturbed_gex_val_pred = np.load(f'../plots/drugseries/GEARS_perturbation_{un_treatment}_{pert_treatment}_{i}.npy')  # .reshape(1, -1) mean prediction
        unperturbed_gex_val = np.load(f'../plots/drugseries/True_un_{un_treatment}_{pert_treatment}_{i}.npy')
        perturbed_gex_val = np.load(f'../plots/drugseries/True_pert_{un_treatment}_{pert_treatment}_{i}.npy')
    except FileNotFoundError as e:
        continue

    print(perturbed_gex_val_pred.shape, unperturbed_gex_val.shape, perturbed_gex_val.shape)

    # Compute metrics
    mse_diff = mean_mse_diff(perturbed_gex_val, perturbed_gex_val_pred)
    pdelta = pearson_delta(unperturbed_gex_val, perturbed_gex_val, perturbed_gex_val_pred)
    if perturbed_gex_val_pred.shape[0] > 1:
        wass_dist = wasserstein_distance(perturbed_gex_val, perturbed_gex_val_pred)
    else:
        wass_dist = float('nan')
    
    # Print
    print('\t'.join([
        f'GEARS',
        f'run{i}',
        f'{pert_treatment.split("_")[1]}',
        f'{":".join(un_treatment.split("_"))}',
        f'{mse_diff:.5f}',
        f'{pdelta:.5f}',
        f'{wass_dist:.5f}',
    ]))

(300, 32738) (2256, 32738) (944, 32738)
GEARS	run0	3hr	DMSO:48hr	0.00931	0.44987	210.18872
(300, 32738) (2256, 32738) (944, 32738)
GEARS	run1	3hr	DMSO:48hr	0.01754	0.43099	210.31021
(300, 32738) (2256, 32738) (944, 32738)
GEARS	run2	3hr	DMSO:48hr	0.02832	0.40760	214.10398
(1, 32738) (2256, 32738) (944, 32738)
GEARS	run3	3hr	DMSO:48hr	0.01134	0.43361	nan
(1, 32738) (2256, 32738) (944, 32738)
GEARS	run4	3hr	DMSO:48hr	0.01314	0.46591	nan
(1, 32738) (2256, 32738) (1059, 32738)
GEARS	run0	6hr	DMSO:48hr	0.01263	0.44304	nan
(1, 32738) (2256, 32738) (1059, 32738)
GEARS	run1	6hr	DMSO:48hr	0.01720	0.44661	nan


KeyboardInterrupt: 

# CPA

In [ ]:
import cpa
import pandas as pd

In [ ]:
# Aggregate sources and targets
combined_treatment_list = collections.defaultdict(list)
for un_treatment, pert_treatment in zip(un_treatment_list, pert_treatment_list):
    combined_treatment_list[un_treatment].append(pert_treatment)
combined_treatment_list = dict(combined_treatment_list)

for i, (un_treatment, pert_treatment_agg) in itertools.product(range(5), combined_treatment_list.items()):
    # Skip if computed
    if os.path.exists(f'../plots/drugseries/CPA_perturbation_{un_treatment}_{pert_treatment}_{i}.npy'):
        continue

    # Seed
    import torch
    torch.manual_seed(42 + i)
    np.random.seed(42 + i)
    torch.set_float32_matmul_precision('high')

    # Read and format
    adata = sc.read_h5ad(f'../plots/drugseries/tram_data_all.h5ad')
    adata.obs['split'] = 'val'
    adata.obs.loc[adata.obs['train'], 'split'] = 'train'
    # # adata = adata[adata.obs['Training']]  # Filter to training
    # adata.obs['cell_type'] = adata.obs['cell_line']
    # conditions = {
    #     un_treatment: 'ctrl',
    #     pert_treatment: 'MAP2K1+MAP2K2'}
    # adata = adata[adata.obs['treatment'].isin(list(conditions.keys()))]
    # adata.obs['condition'] = adata.obs['treatment'].cat.rename_categories(conditions)

    # Perform normalization
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # Prepare data
    cpa.CPA.setup_anndata(
        adata,
        perturbation_key='treatment',
        dosage_key=None,
        control_group=un_treatment,
        batch_key=None,
        is_count_data=True,
        categorical_covariate_keys=['cell_line'],
        # deg_uns_key='rank_genes_groups_cov',
        # deg_uns_cat_key='cov_drug_dose',
        max_comb_len=2)

    # Prepare and train model
    model = cpa.CPA(
        adata=adata,
        split_key='split',
        train_split='train',
        valid_split='val',
        test_split='val',
        n_latent=32,
        seed=42+i)
    model.train(
        max_epochs=2000,
        use_gpu=True,
        batch_size=128,
        early_stopping_patience=10,
        check_val_every_n_epoch=5,
        save_path='../plots/drugseries/cpa/')
    # model.save('../plots/drugseries/cpa/', overwrite=True)

    # Predict perturbations
    model.predict(adata, batch_size=1024)

    # Prepare data
    adata_pert = adata[adata.obs['treatment']==un_treatment].copy()
    for pert_treatment in pert_treatment_agg:
        # Grab mean and std
        counts = adata[(adata.obs['split'] == 'train') * adata.obs['treatment'].isin([un_treatment, pert_treatment])].X.toarray()
        mean = counts.mean(axis=0, keepdims=True)
        std = counts.std(axis=0, keepdims=True)
        std = np.where(std == 0, 1, std)

        # Prepare perturbation
        adata_pert.obs['treatment'] = pert_treatment
        cpa.CPA.setup_anndata(
            adata_pert,
            perturbation_key='treatment',
            dosage_key=None,
            control_group=un_treatment,
            batch_key=None,
            is_count_data=True,
            categorical_covariate_keys=['cell_line'],
            # deg_uns_key='rank_genes_groups_cov',
            # deg_uns_cat_key='cov_drug_dose',
            max_comb_len=2)

        # Filter to val
        common_mask = np.ones(adata_pert.shape[0], dtype=bool)
        # common_mask = ~adata_pert.obs['Training']

        # Get CPA output
        pred = adata_pert.obsm['CPA_pred'].copy()
        np.save(f'../plots/drugseries/CPA_perturbation_{un_treatment}_{pert_treatment}_{i}.npy', (pred[common_mask] - mean) / std)
        # df = pd.DataFrame(pred, columns=adata_pert.var['gene_name'])
        # df['cell_line'] = adata_pert[common_mask].obs['cell_line'].to_numpy()
        # for cell_line, row in df.groupby('cell_line').mean().iterrows():
        #     np.save(f'../plots/drugseries/CPA_{cell_line}_perturbation.npy', row.to_numpy())

In [ ]:
# Load each result sequentially
results = []
for (un_treatment, pert_treatment), i in itertools.product(zip(un_treatment_list, pert_treatment_list), range(5)):
    # Load perturbed result
    try:
        perturbed_gex_val_pred = np.load(f'../plots/drugseries/CPA_perturbation_{un_treatment}_{pert_treatment}_{i}.npy')
        unperturbed_gex_val = np.load(f'../plots/drugseries/True_un_{un_treatment}_{pert_treatment}_{i}.npy')
        perturbed_gex_val = np.load(f'../plots/drugseries/True_pert_{un_treatment}_{pert_treatment}_{i}.npy')
    except FileNotFoundError as e:
        continue

    # Compute metrics
    mse_diff = mean_mse_diff(perturbed_gex_val, perturbed_gex_val_pred)
    pdelta = pearson_delta(unperturbed_gex_val, perturbed_gex_val, perturbed_gex_val_pred)
    wass_dist = wasserstein_distance(perturbed_gex_val, perturbed_gex_val_pred)
    
    # Print
    print('\t'.join([
        f'CPA',
        f'run{i}',
        f'{pert_treatment.split("_")[1]}',
        f'{":".join(un_treatment.split("_"))}',
        f'{mse_diff:.5f}',
        f'{pdelta:.5f}',
        f'{wass_dist:.5f}',
    ]))